In [16]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

df_raw= pd.read_csv("../data/atp_matches_2000_2019_raw.csv")
df_clean = df_raw.copy()

In [17]:
#Observations in the raw dataset
len(df_clean)

61732

In [18]:
#Data copy for cleaning
df_clean = df_raw.copy()

#Remove carpet court surface matches
df_clean = df_clean[df_clean["surface"].isin(["Hard", "Clay", "Grass"])]
df_clean = df_clean.reset_index(drop=True)
df_clean["surface"].value_counts()

#Drop the minute column
df_clean = df_clean.drop(columns=["minutes"])

#Remove Davis cup matches
df_clean = df_clean[df_clean["tourney_level"] != "D"].copy()

#Remove observations with missing values in the winner/loser rank columns
wl_cols = [col for col in df_clean.columns if col.startswith(("w_", "l_"))]
df_clean = df_clean.dropna(subset=wl_cols).copy()

#Chronological time order
df_clean["tourney_date"] = pd.to_datetime(
    df_clean["tourney_date"].astype(str),
    format="%Y%m%d")

#Remove clear height value errors
heights = pd.concat([
    df_clean["winner_ht"],
    df_clean["loser_ht"]])

heights.min(), heights.max()

df_clean = df_clean[
    (df_clean["winner_ht"].between(140, 215)) &
    (df_clean["loser_ht"].between(140, 215))]

#Remove seed and entry columns
df_clean[[
    "winner_seed", "winner_entry",
    "loser_seed", "loser_entry"
]].isna().mean() * 100

df_clean = df_clean.drop(columns=[
    "winner_seed", "winner_entry",
    "loser_seed", "loser_entry"])


#Remove players with A or U handedness
df_clean = df_clean[
    ~df_clean["winner_name"].isin(["Luke Jenssen", "Christopher Koderisch"]) &
    ~df_clean["loser_name"].isin(["Luke Jenssen", "Christopher Koderisch"])
].copy()

In [19]:
#Impute missing ranking values with the player's last known ranking.
df_clean = df_clean.sort_values("tourney_date")

df_clean["winner_rank"] = (
    df_clean
    .groupby("winner_name")["winner_rank"]
    .ffill())

df_clean["loser_rank"] = (
    df_clean
    .groupby("loser_name")["loser_rank"]
    .ffill())


#Impute missing ranking points values with the player's last known ranking points.
df_clean["winner_rank_points"] = (
    df_clean
    .groupby("winner_name")["winner_rank_points"]
    .ffill())

df_clean["loser_rank_points"] = (
    df_clean
    .groupby("loser_name")["loser_rank_points"]
    .ffill())

#If a player has no ranking in any match, it can be assumed that they have not played many matches or their ranking level is low
#Thus, impute these players with lowest ATP ranking (2000) and ATP rank points (0).

df_clean[["winner_rank", "loser_rank"]] = (
    df_clean[["winner_rank", "loser_rank"]].fillna(2000))

df_clean[["winner_rank_points", "loser_rank_points"]] = (
    df_clean[["winner_rank_points", "loser_rank_points"]].fillna(0))

In [20]:
len(df_clean)

53099

In [21]:
#Save the data
df_clean.to_csv("../data/atp_matches_2000_2019_clean.csv", index=False)